# FDA Pathway Predictor — Interactive Testing Notebook

Use this notebook to explore the data and test the ML pipeline step by step.

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 120)

ARTIFACTS = Path('artifacts')
print('Artifacts available:', [f.name for f in ARTIFACTS.glob('*')] if ARTIFACTS.exists() else 'Run pipeline first!')

## Cell 2 — Load & Explore Raw Data

In [ ]:
df_raw = pd.read_csv(ARTIFACTS / 'raw_data.csv')
print(f'Raw data: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')
print(f'\nPathway distribution:\n{df_raw["pathway"].value_counts()}')
print(f'\nColumns: {list(df_raw.columns)}')
df_raw.head(10)

## Cell 3 — Inspect Missing Values & Device Class Issues

In [ ]:
missing_class = df_raw[df_raw['device_class'].isna()]
print(f'Records with missing device_class: {len(missing_class)}')

if len(missing_class) > 0:
    print(f'\nDecision codes for missing device_class:')
    print(missing_class['decision_code'].value_counts().head(10))

    sese_missing = missing_class[missing_class['decision_code'] == 'SESE']
    print(f'\nSESE with missing class: {len(sese_missing)} (can be repaired to Class II)')

## Cell 4 — Load & Explore Clean Data

In [ ]:
df_clean = pd.read_csv(ARTIFACTS / 'clean_data.csv')
print(f'Clean data: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns')
print(f'\nNull counts:\n{df_clean.isnull().sum()}')
print(f'\nDevice class distribution:\n{df_clean["device_class"].value_counts()}')
df_clean.head(10)

## Cell 5 — Inspect Dataset Contract

In [ ]:
with open(ARTIFACTS / 'dataset_contract.json') as f:
    contract = json.load(f)

print(f'Target variable: {contract["target_variable"]}')
print(f'Target classes: {contract["target_classes"]}')
print(f'Total records: {contract["total_records"]}')
print(f'\nRecommended features: {contract["recommended_features"]}')
print(f'\nClass distribution: {contract["constraints"]["class_distribution"]}')

## Cell 6 — Load & Explore Features

In [ ]:
df_features = pd.read_csv(ARTIFACTS / 'features.csv')
print(f'Features: {df_features.shape[0]} rows x {df_features.shape[1]} columns')

feature_cols = [c for c in df_features.columns if c not in ['submission_id', 'pathway', 'pathway_encoded']]
print(f'\n{len(feature_cols)} model features: {feature_cols}')
print(f'\nFeature statistics:')
df_features[feature_cols].describe()

## Cell 7 — Load Model & Encoders

In [ ]:
model = joblib.load(ARTIFACTS / 'model.pkl')
print(f'Model type: {type(model).__name__}')
print(f'Model parameters: {model.get_params()}')

with open(ARTIFACTS / 'label_encoders.json') as f:
    encoders = json.load(f)
print(f'\nPathway encoding: {encoders["pathway"]}')
print(f'Pathway inverse: {encoders["pathway_inverse"]}')

with open(ARTIFACTS / 'model_meta.json') as f:
    meta = json.load(f)
print(f'\nBest model: {meta["best_model"]}')
print(f'Feature columns: {meta["feature_columns"]}')

## Cell 8 — Test a Single Prediction

In [ ]:
test_device = {
    'device_class': 2,
    'advisory_committee_freq': 0.12,
    'advisory_committee_encoded': 3,
    'medical_specialty_freq': 0.12,
    'medical_specialty_encoded': 3,
    'is_us': 1,
    'country_freq': 0.70,
    'decision_year': 2025,
    'decision_month': 6,
    'month_sin': np.sin(2 * np.pi * 6 / 12),
    'month_cos': np.cos(2 * np.pi * 6 / 12),
    'review_days': df_features['review_days'].median(),
    'has_review_days': 1,
    'clearance_type_encoded': 0,
    'third_party': 0,
    'product_code_freq': 0.01,
    'applicant_freq': 0.02,
    'applicant_submission_count': 50,
}

X_test = pd.DataFrame([test_device])
X_test = X_test.reindex(columns=meta['feature_columns'], fill_value=0)

prediction = model.predict(X_test)[0]
probabilities = model.predict_proba(X_test)[0]

pathway_name = encoders['pathway_inverse'][str(prediction)]
print(f'Predicted pathway: {pathway_name}')
print(f'Confidence: {max(probabilities)*100:.1f}%')
print(f'\nProbabilities:')
for class_name, idx in encoders['pathway'].items():
    print(f'  {class_name}: {probabilities[idx]*100:.1f}%')

## Cell 9 — Test Similar Device Lookup

In [ ]:
mask = (df_clean['device_class'] == 2) & (df_clean['advisory_committee'] == 'CV')
similar = df_clean[mask]
print(f'Found {len(similar)} similar devices (Class II, Cardiovascular)')

print(f'\nPathway distribution:')
pathway_pct = (similar['pathway'].value_counts() / len(similar) * 100).round(1)
for pathway, pct in pathway_pct.items():
    print(f'  {pathway}: {pct}%')

print(f'\nDecision codes:')
decision_pct = (similar['decision_code'].value_counts() / len(similar) * 100).round(1)
for code, pct in decision_pct.head(5).items():
    print(f'  {code}: {pct}%')

k510_similar = similar[similar['pathway'] == '510k']
if len(k510_similar) > 0:
    sese_rate = (k510_similar['decision_code'] == 'SESE').mean() * 100
    print(f'\nSESE clearance rate for 510(k): {sese_rate:.1f}%')

## Cell 10 — Test Exemption Check (requires internet)

In [ ]:
import requests

def check_exemption(device_name):
    """Query openFDA classification for exemption status."""
    url = f'https://api.fda.gov/device/classification.json?search=device_name:{device_name}&limit=3'
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            results = r.json().get('results', [])
            for result in results:
                print(f"\nDevice: {result.get('device_name', 'N/A')}")
                print(f"  Product code: {result.get('product_code', 'N/A')}")
                print(f"  Device class: {result.get('device_class', 'N/A')}")
                print(f"  Submission type: '{result.get('submission_type_id', '')}'")
                print(f"  GMP exempt: {result.get('gmp_exempt_flag', 'N/A')}")

                if result.get('device_class') == '1' and result.get('submission_type_id', '').strip() == '':
                    print(f'  EXEMPT -- No premarket submission required')
                else:
                    print(f'  NOT EXEMPT -- Premarket submission required')
            return results
        else:
            print(f'API error: {r.status_code}')
    except Exception as e:
        print(f'Error: {e}')
    return None

print('=== Testing Class I Exempt Device ===')
check_exemption('bandage')

print('\n=== Testing Class III Device ===')
check_exemption('pacemaker')

## Cell 11 — Feature Importance Analysis

In [ ]:
import matplotlib.pyplot as plt

if hasattr(model, 'feature_importances_'):
    importance = pd.Series(
        model.feature_importances_,
        index=meta['feature_columns']
    ).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 6))
    importance.plot(kind='barh', ax=ax, color='#378ADD')
    ax.set_title('Feature Importance -- What Drives the Prediction?')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

    print('\nTop 5 features:')
    for feat, imp in importance.tail(5).iloc[::-1].items():
        print(f'  {feat}: {imp:.4f}')

## Cell 12 — Full Pipeline Test (run all steps)

In [ ]:
print('Running full pipeline...')
from src.flow.main_flow import run_pipeline_without_llm
model, results = run_pipeline_without_llm()

print('\n\nFinal Results:')
for name, metrics in results.items():
    print(f'\n{name}:')
    print(f'  Accuracy:  {metrics["accuracy"]:.4f}')
    print(f'  F1-macro:  {metrics["f1_macro"]:.4f}')
    print(f'  Precision: {metrics["precision_macro"]:.4f}')
    print(f'  Recall:    {metrics["recall_macro"]:.4f}')

print(f'\nAll artifacts:')
for f in sorted(ARTIFACTS.glob('*')):
    print(f'  {f.name}: {f.stat().st_size / 1024:.1f} KB')